This notebook tracks initial MPI creation and decomposition for a MISMIP+ style grid.

It emulates the behaviour of a single user-specified rank decomposition

Acting as if running with `n` processes, with domain decomposition in the x-direction

In [1]:
# Number of processes in x and y directions
px, py = 3, 1

# Number of halo cells on each side of the subgrid
halo::Integer = 2

# Total number of processes
global_size::Integer = px * py

3

Create standard MISMIP+ grid

In [2]:
using WAVI
function MISMIP_PLUS_GRID(;
        nx = 80,
        ny = 10,
    )
    #Grid and boundary conditions
    nσ = 4
    x0 = 0.0
    y0 = -40000.0
    dx = 8000.0
    dy = 8000.0
    h_mask=trues(nx,ny)
    u_iszero = falses(nx+1,ny); u_iszero[1,:].=true
    v_iszero=falses(nx,ny+1); v_iszero[:,1].=true; v_iszero[:,end].=true
    grid = Grid(nx = nx, 
                ny = ny,   
                nσ = nσ, 
                x0 = x0, 
                y0 = y0, 
                dx = dx, 
                dy = dy,
                h_mask = h_mask, 
                u_iszero = u_iszero, 
                v_iszero = v_iszero)
    return grid
end

MISMIP_PLUS_GRID (generic function with 1 method)

This section emulates the creation of an MPI environment like the following:

```julia
    MPI.Init()
    comm = MPI.COMM_WORLD
    rank = MPI.Comm_rank(comm)
    size = MPI.Comm_size(comm)
    @debug "Creating dimensions of $(size) with ($(px), $(py))"

    # Create Virtual Cartesian Topology based on no. of procs in each direction
    dims = MPI.Dims_create(size, (px, py))
    cart_comm = MPI.Cart_create(comm, dims)
    x_coord, y_coord = MPI.Cart_coords(cart_comm)

    # Return the rank of the process in the direction specified
    # Note: MPI_Cart_shift returns `MPI_PROC_NULL` (represented as a negative integer)
    #       if the neighbour is not present
    top = MPI.Cart_shift(cart_comm, 1, -1)[2]
    right = MPI.Cart_shift(cart_comm, 0, 1)[2]
    bottom = MPI.Cart_shift(cart_comm, 1, 1)[2]
    left = MPI.Cart_shift(cart_comm, 0, -1)[2]
```



In [3]:
# Emulate MPI_PROC_NULL (typically -1 in MPI.jl)
const PROC_NULL = -1

function emulate_mpi_creation(global_size::Integer, px::Integer, py::Integer, current_rank::Integer)
    # Calculate coordinates (x, y) for the given rank
    # Note: MPI Cart_create with dims=(px, py) varies the last dimension (y) fastest
    x_coord = div(current_rank, py)  # Column number
    y_coord = current_rank % py      # Row number
    coords = [x_coord, y_coord]

    # Function to calculate the destination rank based on shift direction
    function get_neighbor(direction, displacement)
        # Apply shift
        nx, ny = x_coord, y_coord
        if direction == 0  # X-direction (left/right)
            nx += displacement
        else  # Y-direction (up/down)
            ny += displacement
        end

        # Check boundaries and return destination rank (or PROC_NULL if out of bounds)
        if nx < 0 || nx >= px || ny < 0 || ny >= py
            return PROC_NULL
        else
            return nx * py + ny  # Rank calculation based on column-major order (x*py + y)
        end
    end

    # Emulate Cart_shift for all 4 directions (top, right, bottom, left)
    top    = get_neighbor(1, -1)  # Y-direction, shift -1 (Up)
    right  = get_neighbor(0, 1)   # X-direction, shift +1 (Right)
    bottom = get_neighbor(1, 1)   # Y-direction, shift +1 (Down)
    left   = get_neighbor(0, -1)  # X-direction, shift -1 (Left)

    return coords, top, right, bottom, left
end

# Testing

# # Usage: Emulate rank 0 in a 2x1 grid (global_size=2)
# # (x=0, y=0) -> neighbors should be Right=1, others PROC_NULL
coords, top, right, bottom, left = emulate_mpi_creation(2, 2, 1, 0)

# Usage: Emulate a grid with size 6 (e.g., 2x3 grid) and get neighbors for rank 1
# coords, top, right, bottom, left = emulate_mpi_creation(6, 3, 2, 1)


println("Coords: $(coords)")
println("Neighbors: T:$top, R:$right, B:$bottom, L:$left")


Coords: [0, 0]
Neighbors: T:-1, R:1, B:-1, L:-1


In [4]:
struct MPISpec
    top::Integer
    right::Integer
    bottom::Integer
    left::Integer
    halo::Integer
    global_grid_nx::Integer
    global_grid_ny::Integer
    px::Integer
    py::Integer
    coords::Array{Integer, 1}
    global_size::Integer
    rank::Integer
end

In [5]:
"""
Get the halo size for each direction

Returns (top, right, bottom, left) halo sizes

    Args:
        halo (Integer): Halo size
    Returns:
        Tuple{Int, Int, Int, Int}: Halo size for each direction

"""
function get_halos(spec::MPISpec)::Tuple{Int, Int, Int, Int}
    # Note: MPI_Cart_shift returns `MPI_PROC_NULL` (represented as a negative integer)
    #       if the neighbour is not present
    return spec.top > -1 ? spec.halo : 0,
           spec.right > -1 ? spec.halo : 0,
           spec.bottom > -1 ? spec.halo : 0,
           spec.left > -1 ? spec.halo : 0
end

get_halos

In [6]:
# Get size of local grid
function get_size(spec::MPISpec)::Tuple{Int, Int}
    nx_base = div(spec.global_grid_nx, spec.px)
    ny_base = div(spec.global_grid_ny, spec.py)
    nx_rem  = rem(spec.global_grid_nx, spec.px)
    ny_rem  = rem(spec.global_grid_ny, spec.py)

    # If this rank's x-coord is < remainder, it gets an extra cell
    # This accounts for the fact that the global grid may not be evenly split
    # (e.g. if the global grid is 10x10 and we have 3 ranks in x, then the first
    # rank will have 4 cells in x and the other two ranks will have 3 cells in x)
    local_nx = nx_base + (spec.coords[1] < nx_rem ? 1 : 0)
    local_ny = ny_base + (spec.coords[2] < ny_rem ? 1 : 0)

    halos = get_halos(spec)
    local_nx += halos[4] + halos[2]
    local_ny += halos[1] + halos[3]

    return local_nx, local_ny
end

get_size (generic function with 1 method)

In [7]:
# Get bounds of local grid
function get_bounds(spec::MPISpec)::Tuple{Int, Int, Int, Int}
    # Calculate base local grid size assuming even distribution
    nx_base = div(spec.global_grid_nx, spec.px)
    nx_rem  = rem(spec.global_grid_nx, spec.px)
    ny_base = div(spec.global_grid_ny, spec.py)
    ny_rem  = rem(spec.global_grid_ny, spec.py)

    # Adding extra cell to local grid size if there is a remainder
    local_nx = nx_base + (spec.coords[1] < nx_rem ? 1 : 0)
    local_ny = ny_base + (spec.coords[2] < ny_rem ? 1 : 0)

    # Calculate start index (accounting for previous ranks that took an extra cell)
    # (Rank * BaseSize) + (How many previous ranks took an extra cell) + 1
    x_start = spec.coords[1] * nx_base + min(spec.coords[1], nx_rem) + 1
    y_start = spec.coords[2] * ny_base + min(spec.coords[2], ny_rem) + 1
    
    # Calculate end index
    x_end = x_start + local_nx - 1
    y_end = y_start + local_ny - 1

    halos = get_halos(spec)
    return max(x_start - halos[4], 1),                   # Left: Protect against indices < 1
           min(x_end   + halos[2], spec.global_grid_nx), # Right: Protect against indices > global max
           max(y_start - halos[1], 1),                   # Bottom: Protect against indices < 1
           min(y_end   + halos[3], spec.global_grid_ny)  # Top: Protect against indices > global max
end

get_bounds (generic function with 1 method)

In [8]:
"""
Emulate a single rank decomposition

    Args:
        current_rank (int): The rank to emulate
"""
function emulate_rank(current_rank::Integer)
    grid = MISMIP_PLUS_GRID()

    coords, top, right, bottom, left = emulate_mpi_creation(global_size, px, py, current_rank)
    spec = MPISpec(top, right, bottom, left, halo, grid.nx, grid.ny, px, py, coords, global_size, current_rank)

    th, rh, bh, lh = get_halos(spec)
    nx_local, ny_local = get_size(spec)
    x_start, x_end, y_start, y_end = get_bounds(spec)

    x0_local = grid.x0 + (x_start-1) * grid.dx
    y0_local = grid.y0 + (y_start-1) * grid.dy
    
    @info "[$(current_rank+1)/$(global_size)] - proc $(coords[1]),$(coords[2]) - grid $(nx_local)x$(ny_local)"
    @info "[$(current_rank+1)/$(global_size)] - X [$(x_start):$(x_end)] - Y [$(y_start):$(y_end)] - Centroid $(x0_local),$(y0_local) "
end

emulate_rank

Emulate a single rank from multiple processes

In [9]:
for rank in 0:global_size-1
    emulate_rank(rank)
end

┌ Info: [1/3] - proc 0,0 - grid 29x10
└ @ Main /home/bryald/git/WAVI/WAVI.jl/dev/notebooks/mpi/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:20
┌ Info: [1/3] - X [1:29] - Y [1:10] - Centroid 0.0,-40000.0 
└ @ Main /home/bryald/git/WAVI/WAVI.jl/dev/notebooks/mpi/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:21
┌ Info: [2/3] - proc 1,0 - grid 31x10
└ @ Main /home/bryald/git/WAVI/WAVI.jl/dev/notebooks/mpi/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:20
┌ Info: [2/3] - X [26:56] - Y [1:10] - Centroid 200000.0,-40000.0 
└ @ Main /home/bryald/git/WAVI/WAVI.jl/dev/notebooks/mpi/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:21
┌ Info: [3/3] - proc 2,0 - grid 28x10
└ @ Main /home/bryald/git/WAVI/WAVI.jl/dev/notebooks/mpi/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:20
┌ Info: [3/3] - X [53:80] - Y [1:10] - Centroid 416000.0,-40000.0 
└ @ Main /home/bryald/git/WAVI/WAVI.jl/dev/notebooks/mpi/jl_n